# LeetCode 3944 — Minimum Operations to Make Array Modulo Alternating II bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-make-array-modulo-alternating-ii/
- Độ khó: **Hard**
- Chủ đề: Số dư (modulo), khoảng cách vòng tròn, đếm theo số dư, mảng hiệu (difference array) vòng tròn, chọn hai phần tử khác nhau

> **Bài Premium: đề bài viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**
> Bài Premium không có skeleton công khai. Tên phương thức `minOperations(self, nums, k)` được suy luận — **cần đối chiếu** lại chữ ký trên LeetCode khi nộp.

Skeleton (suy luận):

```python
class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Mảng được gọi là **"modulo alternating"** nếu tồn tại hai số **khác nhau** `x, y ∈ [0, k)` sao cho mọi phần tử ở chỉ số **chẵn** có `nums[i] % k == x` và mọi phần tử ở chỉ số **lẻ** có `nums[i] % k == y`. Mỗi thao tác tăng hoặc giảm một phần tử đi 1. Tìm số thao tác tối thiểu.

Điểm khó của phiên bản II: `n, k` lên tới `10^5`, nên **không thể** thử mọi cặp `(x, y)` (`k²` cặp) rồi cộng chi phí (`n` mỗi cặp). Ta phải:

- Tính chi phí đưa **toàn bộ nhóm chẵn** về số dư `x` — gọi là `E[x]` — và nhóm lẻ về `y` — `O[y]` — cho **mọi** `x, y` trong tổng `O(n + k)`.
- Chọn cặp `x ≠ y` làm cực tiểu `E[x] + O[y]` trong `O(k)` bằng "hai giá trị nhỏ nhất".

Sau notebook này bạn sẽ:

1. Biết chi phí đưa `v` về số dư `t` là **khoảng cách vòng tròn** `min((v−t) mod k, (t−v) mod k)`.
2. Gom phần tử theo số dư `v mod k` để mọi thứ chỉ còn phụ thuộc số dư.
3. Tính `E[t]` cho mọi `t` trong `O(n + k)` bằng **mảng hiệu vòng tròn** (độ dốc của hàm khoảng cách vòng tròn).
4. Chọn `x ≠ y` tối ưu bằng hai giá trị nhỏ nhất của `O` (không cần `O(k²)`).
5. Viết 3 lời giải (vét cạn theo định nghĩa, tổng hợp chi phí `O(n·k)`, mảng hiệu `O(n + k)`) và đối chiếu bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums` và `k`. Mỗi thao tác: tăng hoặc giảm một phần tử đi 1. Mảng là *modulo alternating* nếu có hai số nguyên **phân biệt** `x, y` với `0 ≤ x, y < k` sao cho:

- mọi chỉ số chẵn `i`: `nums[i] % k == x`;
- mọi chỉ số lẻ `i`: `nums[i] % k == y`.

Trả về số thao tác tối thiểu.

**Ví dụ 1:** `nums = [1,4,2,8]`, `k = 3` → `2` (chọn `x = 1`, `y = 2`: tăng `4` thành `5`, giảm `2` thành `1`).

**Ví dụ 2:** `nums = [1,1,1]`, `k = 3` → `1` (tăng phần tử giữa thành `2`; `x = 1`, `y = 2`).

**Ví dụ 3:** `nums = [6,7,8]`, `k = 2` → `0` (đã có `x = 0`, `y = 1`).

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^9`, `2 ≤ k ≤ 10^5`.

In [1]:
import random, time
from collections import Counter

EXAMPLES = [
    (([1, 4, 2, 8], 3), 2),
    (([1, 1, 1], 3), 1),
    (([6, 7, 8], 2), 0),
]
for args, exp in EXAMPLES:
    print(args, "-> kỳ vọng", exp)

([1, 4, 2, 8], 3) -> kỳ vọng 2
([1, 1, 1], 3) -> kỳ vọng 1
([6, 7, 8], 2) -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Toán tử `%` (số dư) trong Python

`a % k` luôn nằm trong `[0, k)` (kể cả khi `a` âm) — rất tiện cho khoảng cách vòng tròn.

In [2]:
print(14 % 3)       # 2
print(-1 % 3)       # 2  (Python: kết quả không âm)
print((1 - 2) % 3)  # 2
print((2 - 1) % 3)  # 1

2
2
2
1


### 3.2. Chi phí đưa `v` về số dư `t`: khoảng cách vòng tròn

Muốn `v` có số dư `t` ta phải đổi `v` thành một số `w ≡ t (mod k)`; tốn `|v − w|` thao tác. Các số `w` có số dư `t` nằm cách nhau `k` đơn vị trên trục số, nên `w` gần `v` nhất cách `v`:

- **đi lên:** `(t − v) mod k` bước, hoặc
- **đi xuống:** `(v − t) mod k` bước.

Hai số này cộng lại bằng `k` (nếu `v ≢ t`), nên chi phí là `min((t − v) mod k, (v − t) mod k)` — chính là khoảng cách trên **vòng tròn** có `k` điểm `0..k−1` giữa `r = v mod k` và `t`.

Ví dụ `k = 5`: từ số dư `4` về số dư `0` chỉ tốn `1` (đi lên `4 → 5`), không phải `4`.

In [3]:
def circ(r, t, k):
    d = (r - t) % k
    return min(d, k - d)

k = 5
print([circ(4, t, k) for t in range(k)])     # từ 4: [1, 2, 2, 1, 0]
print([circ(0, t, k) for t in range(k)])     # từ 0: [0, 1, 2, 2, 1]
# kiểm chứng định nghĩa: số bước thật sự để đổi v thành số có số dư t
def real_cost(v, t, k):
    return min(abs(v - w) for w in range(v - 2 * k, v + 2 * k + 1) if w % k == t)
assert all(real_cost(v, t, 7) == circ(v % 7, t, 7) for v in range(1, 40) for t in range(7))
print("khoảng cách vòng tròn khớp chi phí thật")

[1, 2, 2, 1, 0]
[0, 1, 2, 2, 1]
khoảng cách vòng tròn khớp chi phí thật


### 3.3. Gom theo số dư

Chi phí chỉ phụ thuộc `r = v mod k`. Với mỗi nhóm (chẵn/lẻ) ta đếm `cnt[r]` = số phần tử có số dư `r`. Có `k` số dư nên bảng đếm dài `k`, và cả hai nhóm dùng chung kỹ thuật.

In [4]:
nums, k = [1, 4, 2, 8], 3
cntE = [0] * k; cntO = [0] * k
for i, v in enumerate(nums):
    (cntE if i % 2 == 0 else cntO)[v % k] += 1
print("chỉ số chẵn:", nums[0::2], "-> cntE =", cntE)
print("chỉ số lẻ  :", nums[1::2], "-> cntO =", cntO)

chỉ số chẵn: [1, 2] -> cntE = [0, 1, 1]
chỉ số lẻ  : [4, 8] -> cntO = [0, 1, 1]


### 3.4. Hai giá trị nhỏ nhất kèm chỉ số

Cần `min` của `E[x] + O[y]` với `x ≠ y`. Với mỗi `x`, cần `min` của `O[y]` trên `y ≠ x`: nếu `x` không phải chỉ số đạt `O` nhỏ nhất thì lấy `O` nhỏ nhất, ngược lại lấy giá trị nhỏ thứ hai. Do đó chỉ cần biết **hai giá trị nhỏ nhất của `O` và chỉ số của giá trị nhỏ nhất**.

In [5]:
O = [5, 2, 9, 2, 7]
order = sorted(range(len(O)), key=lambda i: O[i])
a, b = order[0], order[1]
print("nhỏ nhất:", O[a], "ở", a, "; nhỏ thứ hai:", O[b], "ở", b)
for x in range(len(O)):
    best_other = O[b] if x == a else O[a]
    assert best_other == min(O[y] for y in range(len(O)) if y != x)
print("quy tắc 'nếu x trùng chỉ số nhỏ nhất thì lấy nhỏ thứ hai' đúng")

nhỏ nhất: 2 ở 1 ; nhỏ thứ hai: 2 ở 3
quy tắc 'nếu x trùng chỉ số nhỏ nhất thì lấy nhỏ thứ hai' đúng


### 3.5. Mảng hiệu (difference array) trên vòng tròn

Để cộng `+1` cho một **đoạn** vị trí `[l, r)` của mảng mà không lặp qua từng vị trí: `D[l] += 1`, `D[r] −= 1`; sau đó **tổng tiền tố** của `D` cho giá trị đã cộng tại từng vị trí. Nếu đoạn **quấn vòng** (vượt qua cuối mảng) thì tách thành hai đoạn: `[l, k)` và `[0, r − k)`.

In [6]:
def circular_range_add(D, k, l, length, val):
    # cộng val cho các vị trí l, l+1, ..., l+length-1 (mod k); D có độ dài k+1
    if length <= 0:
        return
    r = l + length
    if r <= k:
        D[l] += val; D[r] -= val
    else:
        D[l] += val; D[k] -= val
        D[0] += val; D[r - k] -= val

k = 6
D = [0] * (k + 1)
circular_range_add(D, k, 4, 4, +1)        # vị trí 4,5,0,1
pref, run = [], 0
for i in range(k):
    run += D[i]
    pref.append(run)
print(pref)    # [1, 1, 0, 0, 1, 1]

[1, 1, 0, 0, 1, 1]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `nums = [1, 4, 2, 8]`, `k = 3`.

- Chỉ số chẵn: `1, 2` → số dư `1, 2`. Chỉ số lẻ: `4, 8` → số dư `1, 2`.
- Bảng `E[t]` (đưa cả nhóm chẵn về `t`): `E[0] = circ(1,0)+circ(2,0) = 1+1 = 2`; `E[1] = 0+1 = 1`; `E[2] = 1+0 = 1`.
- Bảng `O[t]` giống hệt: `O = [2, 1, 1]`.
- Cặp `x ≠ y` tốt nhất: `x = 1, y = 2` → `E[1] + O[2] = 1 + 1 = 2` ✓. (Cặp `x = y = 1` tốn `1 + 1 = 2` nhưng **bị cấm**; ở ví dụ này tình cờ bằng.)

Với **Ví dụ 2**: `[1,1,1]`: `E` ứng với số dư `1, 1` → `E = [2, 0, 2]`; `O` từ số dư `1` → `O = [1, 0, 1]`. Chọn `x = y = 1` cho `0` nhưng bị cấm; `x = 1, y = 0` → `0 + 1 = 1` ✓.

**Ý tưởng:** chỉ cần hai bảng `E` và `O`, rồi chọn `x ≠ y`. Vấn đề còn lại: tính `E`, `O` đủ nhanh.

In [7]:
def tables_direct(nums, k):
    # E[t], O[t] theo định nghĩa: cộng khoảng cách vòng tròn của từng phần tử
    E = [0] * k; O = [0] * k
    for i, v in enumerate(nums):
        r = v % k
        T = E if i % 2 == 0 else O
        for t in range(k):
            T[t] += circ(r, t, k)
    return E, O

print(tables_direct([1, 4, 2, 8], 3))
print(tables_direct([1, 1, 1], 3))

([2, 1, 1], [2, 1, 1])
([2, 0, 2], [1, 0, 1])


## 5. Cách 1 — Brute Force theo đúng định nghĩa

**Ý tưởng.** Thử mọi cặp `x ≠ y`. Với mỗi cặp, mỗi phần tử tính chi phí thật bằng cách tìm số `w ≡ (x hoặc y)` gần nhất (không dùng công thức vòng tròn, để làm chuẩn độc lập). Lấy `min`.

In [8]:
def brute_definition(nums, k):
    best = None
    for x in range(k):
        for y in range(k):
            if x == y:
                continue
            total = 0
            for i, v in enumerate(nums):
                t = x if i % 2 == 0 else y
                total += min(abs(v - w) for w in range(v - 2 * k, v + 2 * k + 1) if w % k == t)
            if best is None or total < best:
                best = total
    return best

for args, exp in EXAMPLES:
    got = brute_definition(*args)
    print(args, "->", got)
    assert got == exp

([1, 4, 2, 8], 3) -> 2
([1, 1, 1], 3) -> 1
([6, 7, 8], 2) -> 0


**Độ phức tạp.** `O(k² · n · k)` nếu tìm `w` bằng cách duyệt như trên (ta chỉ dùng nó như bằng chứng độc lập, input rất nhỏ). Dùng công thức vòng tròn thì `O(k² · n)`. Với `n = k = 10^5` là `10^15` — không thể.

**Nút thắt.** (1) Lặp qua mọi cặp `(x, y)` thay vì xét hai nhóm độc lập; (2) mỗi cặp lại quét cả mảng.

## 6. Bước đột phá

**Quan sát 1 — hai nhóm độc lập, chỉ ràng buộc bởi `x ≠ y`.** Tổng chi phí `= E[x] + O[y]`; nhóm chẵn không quan tâm `y` và ngược lại. Vậy ta tính hai bảng `E`, `O` riêng, rồi chọn cặp.

**Quan sát 2 — chọn cặp `x ≠ y` trong `O(k)`.** Với mỗi `x`, `y` tốt nhất là chỉ số nhỏ nhất của `O` khác `x` (mục 3.4). Do đó `answer = min_x (E[x] + (O nhỏ nhất nếu x ≠ a, ngược lại O nhỏ thứ hai))`.

**Quan sát 3 — `E[t]` là tổng của các "hàm khoảng cách vòng tròn" nên có độ dốc đơn giản.** Với một phần tử số dư `r`, hàm `t ↦ circ(r, t, k)` bằng `0` tại `t = r`, tăng `+1` mỗi bước trong nửa vòng đầu, rồi giảm `−1` mỗi bước trong nửa vòng sau (có thể có một bước phẳng ở giữa nếu `k` lẻ). Cụ thể, đặt `P = k // 2` và `j = (t − r) mod k`, **độ dốc** `s(t) = circ(t+1) − circ(t)` bằng:

- `+1` khi `0 ≤ j < P`;
- `−1` khi `j ≥ k − P` (`P` vị trí cuối);
- `0` ở vị trí còn lại (chỉ tồn tại khi `k` lẻ: `j = P`).

Kiểm tra: `k = 4`, `r = 0`: `circ = [0, 1, 2, 1]`, độ dốc `[+1, +1, −1, −1]` ✓. `k = 5`: `circ = [0, 1, 2, 2, 1]`, độ dốc `[+1, +1, 0, −1, −1]` ✓.

Tổng độ dốc của mọi phần tử cho `S[t] = E[t+1] − E[t]`. Mỗi **nhóm số dư** `r` (có `cnt[r]` phần tử) đóng góp `+cnt[r]` trên đoạn vòng tròn dài `P` bắt đầu ở `r`, và `−cnt[r]` trên đoạn dài `P` bắt đầu ở `(r + k − P) mod k`. Dùng mảng hiệu vòng tròn nên mỗi nhóm tốn `O(1)`.

**Cuối cùng:** `E[0] = Σ cnt[r] · min(r, k − r)` (tính trực tiếp, `O(k)`), rồi `E[t+1] = E[t] + S[t]`. Toàn bộ `O(n + k)`.

In [9]:
# Kiểm tra độ dốc cho một phần tử: P = k//2, +1 trên P vị trí đầu, -1 trên P vị trí cuối, 0 ở giữa nếu k lẻ
def slope_formula(r, t, k):
    P = k // 2
    j = (t - r) % k
    if j < P:
        return 1
    if j >= k - P:
        return -1
    return 0

for k in (2, 3, 4, 5, 6, 7, 10):
    for r in range(k):
        for t in range(k):
            assert circ(r, (t + 1) % k, k) - circ(r, t, k) == slope_formula(r, t, k), (k, r, t)
print("công thức độ dốc đúng cho k = 2..10")

công thức độ dốc đúng cho k = 2..10


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tổng hợp `E`, `O` bằng công thức vòng tròn, `O(n·k)` (hoặc `O(D·k)`)

**Giải thích từng đoạn.**

1. Đếm `cntE[r]`, `cntO[r]` theo số dư (`O(n)`).
2. `E[t] = Σ_r cntE[r] · circ(r, t)` — duyệt các số dư có mặt, mỗi `t` một lần: `O(D·k)` với `D` là số số dư khác nhau (`D ≤ min(n, k)`).
3. Chọn cặp `x ≠ y` bằng hai giá trị nhỏ nhất của `O` (mục 3.4).

Cách này nhanh hơn Cách 1 rất nhiều nhưng vẫn cỡ `10^10` khi `D = k = 10^5`.

In [10]:
def pick_best_pair(E, O):
    k = len(E)
    order = sorted(range(k), key=lambda i: O[i])
    a, b = order[0], order[1]
    best = None
    for x in range(k):
        o = O[b] if x == a else O[a]
        if best is None or E[x] + o < best:
            best = E[x] + o
    return best

def tables_by_residue(nums, k):
    cnt = [[0] * k, [0] * k]
    for i, v in enumerate(nums):
        cnt[i & 1][v % k] += 1
    res = []
    for p in range(2):
        T = [0] * k
        present = [(r, c) for r, c in enumerate(cnt[p]) if c]
        for t in range(k):
            T[t] = sum(c * circ(r, t, k) for r, c in present)
        res.append(T)
    return res

def solve_by_residue(nums, k):
    E, O = tables_by_residue(nums, k)
    return pick_best_pair(E, O)

for args, exp in EXAMPLES:
    got = solve_by_residue(*args)
    print(args, "->", got)
    assert got == exp

([1, 4, 2, 8], 3) -> 2
([1, 1, 1], 3) -> 1
([6, 7, 8], 2) -> 0


**Độ phức tạp:** `O(n + D·k + k log k)` (có thể thay sort bằng một vòng tìm hai nhỏ nhất `O(k)`); với `D ≈ k` là `O(k²)`. **Không gian:** `O(k)`.

### 7.2. Cách 3 — Mảng hiệu vòng tròn, `O(n + k)`

**Giải thích từng đoạn.**

1. Đếm `cnt[r]` cho từng nhóm.
2. `E[0] = Σ cnt[r] · min(r, k − r)`.
3. Dựng mảng hiệu `D` cho độ dốc `S`: với mỗi `r` có `cnt[r] > 0`:
   - `+cnt[r]` trên đoạn vòng tròn bắt đầu `r`, dài `P = k // 2`;
   - `−cnt[r]` trên đoạn vòng tròn bắt đầu `(r + k − P) mod k`, dài `P`.
4. `S` = tổng tiền tố của `D`; `E[t+1] = E[t] + S[t]`.
5. Hai nhóm dùng cùng một hàm `table`; cuối cùng `pick_best_pair`.

In [11]:
def table_from_counts(cnt, k):
    P = k // 2
    D = [0] * (k + 1)
    e0 = 0
    for r, c in enumerate(cnt):
        if not c:
            continue
        e0 += c * min(r, k - r)
        circular_range_add(D, k, r, P, +c)
        circular_range_add(D, k, (r + k - P) % k, P, -c)
    E = [0] * k
    E[0] = e0
    slope = 0
    for t in range(k - 1):
        slope += D[t]
        E[t + 1] = E[t] + slope
    return E

def solve_diff(nums, k):
    cnt = [[0] * k, [0] * k]
    for i, v in enumerate(nums):
        cnt[i & 1][v % k] += 1
    E = table_from_counts(cnt[0], k)
    O = table_from_counts(cnt[1], k)
    return pick_best_pair(E, O)

# kiểm tra bảng E, O khớp tính trực tiếp
rng = random.Random(0)
for _ in range(100):
    k = rng.randint(2, 12)
    nums = [rng.randint(1, 60) for _ in range(rng.randint(1, 10))]
    Ed, Od = tables_direct(nums, k)
    cnt = [[0] * k, [0] * k]
    for i, v in enumerate(nums):
        cnt[i & 1][v % k] += 1
    assert table_from_counts(cnt[0], k) == Ed and table_from_counts(cnt[1], k) == Od, (nums, k)
print("bảng E, O bằng mảng hiệu khớp tính trực tiếp trên 100 input")

for args, exp in EXAMPLES:
    got = solve_diff(*args)
    print(args, "->", got)
    assert got == exp

bảng E, O bằng mảng hiệu khớp tính trực tiếp trên 100 input
([1, 4, 2, 8], 3) -> 2
([1, 1, 1], 3) -> 1
([6, 7, 8], 2) -> 0


**Độ phức tạp:** `O(n + k)` thời gian, `O(k)` không gian. Với `n = k = 10^5` chỉ vài trăm nghìn phép.

Lưu ý: ở vòng `for t in range(k - 1)` ta cộng dồn `slope += D[t]` rồi mới dùng — `slope` tại bước `t` chính là `S[t] = E[t+1] − E[t]`.

## 8. TRACE từng bước

Trace Cách 3 với `nums = [1, 4, 2, 8, 5]`, `k = 5` (`P = 2`).

Nhóm chẵn: `nums[0], nums[2], nums[4] = 1, 2, 5` → số dư `1, 2, 0`, `cnt = [1, 1, 1, 0, 0]`.

- `E[0] = min(0,5)·1 + min(1,4)·1 + min(2,3)·1 = 0 + 1 + 2 = 3`.
- Mỗi số dư `r` cộng `+1` vào `t ∈ {r, r+1}` và `−1` vào `t ∈ {r+3, r+4}` (vì `(r + k − P) = r + 3`), tính theo mod 5.

Độ dốc `S[t]` (cộng ba nhóm):

| `t` | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| từ `r=0` | +1 | +1 | 0 | −1 | −1 |
| từ `r=1` | −1 | +1 | +1 | 0 | −1 |
| từ `r=2` | −1 | −1 | +1 | +1 | 0 |
| `S[t]` | −1 | +1 | +2 | 0 | −2 |

`E = [3, 2, 3, 5, 5]`. Kiểm tra `E[1]`: `circ(1,1)+circ(2,1)+circ(0,1) = 0+1+1 = 2` ✓. Hãy chạy trace để xem từng bước.

In [12]:
def trace_diff(cnt, k):
    P = k // 2
    print(f"k={k}, P={P}, cnt={cnt}")
    D = [0] * (k + 1)
    e0 = 0
    for r, c in enumerate(cnt):
        if not c:
            continue
        e0 += c * min(r, k - r)
        circular_range_add(D, k, r, P, +c)
        circular_range_add(D, k, (r + k - P) % k, P, -c)
        print(f"  số dư r={r} (x{c}): +{c} trên [{r}..{(r + P - 1) % k}], -{c} trên [{(r + k - P) % k}..{(r + k - 1) % k}]  -> D={D}")
    print("E[0] =", e0)
    E = [0] * k
    E[0] = e0
    slope = 0
    for t in range(k - 1):
        slope += D[t]
        E[t + 1] = E[t] + slope
        print(f"  t={t}: slope S[{t}] = {slope:+d}  -> E[{t + 1}] = {E[t + 1]}")
    return E

cnt = [0] * 5
for v in [1, 2, 5]:
    cnt[v % 5] += 1
E = trace_diff(cnt, 5)
exp_E = [sum(circ(v % 5, t, 5) for v in [1, 2, 5]) for t in range(5)]
print("E =", E, " trực tiếp:", exp_E)
assert E == exp_E == [3, 2, 3, 5, 5]

k=5, P=2, cnt=[1, 1, 1, 0, 0]
  số dư r=0 (x1): +1 trên [0..1], -1 trên [3..4]  -> D=[1, 0, -1, -1, 0, 1]
  số dư r=1 (x1): +1 trên [1..2], -1 trên [4..0]  -> D=[0, 2, -1, -2, -1, 2]
  số dư r=2 (x1): +1 trên [2..3], -1 trên [0..1]  -> D=[-1, 2, 1, -2, -2, 2]
E[0] = 3
  t=0: slope S[0] = -1  -> E[1] = 2
  t=1: slope S[1] = +1  -> E[2] = 3
  t=2: slope S[2] = +2  -> E[3] = 5
  t=3: slope S[3] = +0  -> E[4] = 5
E = [3, 2, 3, 5, 5]  trực tiếp: [3, 2, 3, 5, 5]


## 9. Code nộp LeetCode

Gộp Cách 3 thành `class Solution` độc lập. Tên phương thức suy luận — **cần đối chiếu** trên LeetCode.

In [13]:
from typing import List

class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        P = k // 2
        cnt = [[0] * k, [0] * k]
        for i, v in enumerate(nums):
            cnt[i & 1][v % k] += 1

        def build(c):
            # E[t] = tổng khoảng cách vòng tròn từ mọi phần tử (theo số dư) tới số dư t
            D = [0] * (k + 1)
            e0 = 0
            for r in range(k):
                m = c[r]
                if not m:
                    continue
                e0 += m * (r if r < k - r else k - r)
                # +m trên P vị trí bắt đầu từ r (quấn vòng)
                end = r + P
                if end <= k:
                    D[r] += m; D[end] -= m
                else:
                    D[r] += m; D[k] -= m; D[0] += m; D[end - k] -= m
                # -m trên P vị trí bắt đầu từ r + k - P (mod k)
                st = (r + k - P) % k
                end = st + P
                if end <= k:
                    D[st] -= m; D[end] += m
                else:
                    D[st] -= m; D[k] += m; D[0] -= m; D[end - k] += m
            E = [0] * k
            E[0] = e0
            slope = 0
            for t in range(k - 1):
                slope += D[t]
                E[t + 1] = E[t] + slope
            return E

        E = build(cnt[0])
        O = build(cnt[1])
        # hai giá trị nhỏ nhất của O (kèm chỉ số của giá trị nhỏ nhất)
        a = min(range(k), key=O.__getitem__)
        second = min(O[y] for y in range(k) if y != a)
        best = None
        for x in range(k):
            cur = E[x] + (second if x == a else O[a])
            if best is None or cur < best:
                best = cur
        return best

sol = Solution()
for args, exp in EXAMPLES:
    got = sol.minOperations(*args)
    print(args, "->", got)
    assert got == exp

([1, 4, 2, 8], 3) -> 2
([1, 1, 1], 3) -> 1
([6, 7, 8], 2) -> 0


## 10. Vì sao thuật toán đúng

1. **Tách hai nhóm.** Với `(x, y)` cố định, tổng thao tác tối thiểu là tổng độc lập của từng phần tử (mỗi phần tử chỉ cần đạt số dư đích của nhóm mình) nên bằng `E[x] + O[y]`; ràng buộc duy nhất giữa hai nhóm là `x ≠ y`.
2. **Chi phí từng phần tử là khoảng cách vòng tròn.** Các số cùng số dư `t` cách nhau đúng `k` nên số gần `v` nhất cách `v` đúng `min((t−v) mod k, (v−t) mod k)` thao tác (mục 3.2). Số này không bao giờ cần làm `v` âm: đi xuống tối đa `v mod k` bước thì dừng ở bội của `k` không âm.
3. **Độ dốc.** Mỗi hàm `circ(r, ·, k)` có độ dốc `+1` ở `P = ⌊k/2⌋` vị trí đầu, `−1` ở `P` vị trí cuối, `0` ở vị trí giữa nếu `k` lẻ (đã kiểm chứng mọi `k = 2..10` bằng code ở mục 6). Cộng theo nhóm số dư cho `S`; `E[t+1] = E[t] + S[t]` đúng theo định nghĩa độ dốc; `E[0]` tính trực tiếp.
4. **Chọn cặp.** Với `x` cố định `min_{y ≠ x} O[y]` bằng `O` nhỏ nhất nếu `x` không phải một chỉ số đạt nhỏ nhất, ngược lại bằng giá trị nhỏ thứ hai (xét theo vị trí, nên nếu có nhiều chỉ số cùng đạt nhỏ nhất thì giá trị nhỏ thứ hai bằng nhỏ nhất — đúng). Vì `k ≥ 2` nên luôn có `y ≠ x`.

Bài test mục 12 đối chiếu với vét cạn theo định nghĩa trên nhiều input ngẫu nhiên.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`.

| Bước | Thời gian |
|---|---|
| Đếm số dư theo nhóm | `O(n)` |
| Dựng mảng hiệu và `E` (mỗi nhóm) | `O(k)` (mỗi số dư có mặt tốn `O(1)`) |
| Chọn cặp `x ≠ y` | `O(k)` |

**Tổng:** `O(n + k)` thời gian, `O(k)` không gian. Với `n = k = 10^5`: cỡ vài trăm nghìn phép. Giá trị đáp án tối đa cỡ `n · k / 2 = 5·10^9` — Python không tràn số.

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, `k = 2`, mọi phần tử bằng nhau, `k` lớn hơn `n`), phản ví dụ, và đối chiếu ngẫu nhiên giữa 4 cách có seed.

In [14]:
sol = Solution()
for args, exp in EXAMPLES:
    assert sol.minOperations(*args) == exp

assert sol.minOperations([5], 2) == 0                      # n=1: chỉ nhóm chẵn, y tuỳ ý khác x
assert sol.minOperations([7, 7], 2) == 1                   # cùng số dư 1: phải đổi một phần tử
assert sol.minOperations([4, 4, 4, 4], 4) == 2             # E cần x=0 (0 chi phí), O cần y!=0 -> mỗi phần tử lẻ tốn 1 => 2
print("edge cases OK")

edge cases OK


In [15]:
# Đối chiếu: vét cạn theo định nghĩa (nhỏ) và các cách nhanh
rng = random.Random(3944)
for _ in range(120):
    k = rng.randint(2, 7)
    n = rng.randint(1, 7)
    nums = [rng.randint(1, 30) for _ in range(n)]
    exp = brute_definition(nums, k)
    assert solve_by_residue(nums, k) == exp, (nums, k)
    assert solve_diff(nums, k) == exp, (nums, k)
    assert sol.minOperations(nums, k) == exp, (nums, k)
print("120 test nhỏ khớp vét cạn theo định nghĩa")

for _ in range(300):
    k = rng.randint(2, 40)
    n = rng.randint(1, 60)
    nums = [rng.randint(1, 10**9) for _ in range(n)]
    a = solve_by_residue(nums, k)
    assert a == solve_diff(nums, k) == sol.minOperations(nums, k), (nums, k)
print("300 test vừa: Cách 2 = Cách 3 = Solution")

# thời gian lớn nhất
n = k = 10**5
nums = [rng.randint(1, 10**9) for _ in range(n)]
t0 = time.time()
r = sol.minOperations(nums, k)
print("n = k = 1e5 ->", r, f"({time.time() - t0:.2f}s)")

120 test nhỏ khớp vét cạn theo định nghĩa
300 test vừa: Cách 2 = Cách 3 = Solution


n = k = 1e5 -> 2493268329 (0.08s)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên điều kiện `x ≠ y`

**Code sai:** `min(E) + min(O)`.
**Vì sao sai:** nếu hai chỉ số nhỏ nhất trùng nhau thì cặp bị cấm. Ví dụ `[1,1,1]`, `k = 3`: `min(E) = 0` (tại `1`), `min(O) = 0` (tại `1`) → `0`, nhưng đáp án là `1`.
**Code đúng:** với mỗi `x`, lấy `O` nhỏ nhất **khác `x`**.

In [16]:
def wrong_no_distinct(nums, k):
    cnt = [[0] * k, [0] * k]
    for i, v in enumerate(nums):
        cnt[i & 1][v % k] += 1
    E, O = tables_by_residue(nums, k)
    return min(E) + min(O)              # SAI: có thể x == y

print("sai :", wrong_no_distinct([1, 1, 1], 3), " đúng:", Solution().minOperations([1, 1, 1], 3))

sai : 0  đúng: 1


### Lỗi 2 — Dùng khoảng cách tuyến tính `|r − t|` thay vì vòng tròn

**Code sai:** `cost = abs(v % k - t)`.
**Vì sao sai:** từ số dư `k−1` lên số dư `0` chỉ tốn `1` (qua bội của `k`), không phải `k−1`.
**Code đúng:** `min(d, k − d)` với `d = (r − t) % k`.

In [17]:
def wrong_linear(nums, k):
    best = None
    for x in range(k):
        for y in range(k):
            if x != y:
                tot = sum(abs(v % k - (x if i % 2 == 0 else y)) for i, v in enumerate(nums))   # SAI
                best = tot if best is None else min(best, tot)
    return best

rng = random.Random(77)
for _ in range(500):
    k = rng.randint(3, 7); n = rng.randint(2, 6)
    nums = [rng.randint(1, 30) for _ in range(n)]
    if wrong_linear(nums, k) != brute_definition(nums, k):
        print("phản ví dụ:", nums, k, "sai:", wrong_linear(nums, k), "đúng:", brute_definition(nums, k))
        break

phản ví dụ: [16, 18, 20, 8] 3 sai: 3 đúng: 2


### Lỗi 3 — Chỉ cho phép tăng (hoặc chỉ giảm)

**Code sai:** `cost = (t − r) % k`.
**Vì sao sai:** đề cho phép cả tăng lẫn giảm; chọn hướng ngắn hơn.
**Code đúng:** `min((t − r) % k, (r − t) % k)`.

In [18]:
def wrong_only_up(nums, k):
    best = None
    for x in range(k):
        for y in range(k):
            if x != y:
                tot = sum((((x if i % 2 == 0 else y) - v) % k) for i, v in enumerate(nums))     # SAI
                best = tot if best is None else min(best, tot)
    return best

print("sai :", wrong_only_up([1, 4, 2, 8], 3), " đúng:", Solution().minOperations([1, 4, 2, 8], 3))

sai : 3  đúng: 2


### Lỗi 4 — Tính `E[t]` cho mọi `t` bằng vòng lặp lồng nhau (TLE)

`E[t] = Σ_i circ(...)` cho `t = 0..k−1` và mọi `i` là `O(n·k) = 10^10`. Hãy dùng mảng hiệu (Cách 3).

In [19]:
rng = random.Random(5)
n = k = 1500
nums = [rng.randint(1, 10**9) for _ in range(n)]
t0 = time.time(); a = solve_by_residue(nums, k); t1 = time.time()
b = Solution().minOperations(nums, k); t2 = time.time()
assert a == b
print(f"n=k=1500: tính E theo từng số dư {t1 - t0:.2f}s, mảng hiệu {t2 - t1:.3f}s -> chênh lệch tăng rất nhanh khi n, k lớn")

n=k=1500: tính E theo từng số dư 0.23s, mảng hiệu 0.001s -> chênh lệch tăng rất nhanh khi n, k lớn


### Lỗi 5 — Lấy `nums[i] − t` trực tiếp, quên `% k` trước

`nums[i]` lên tới `10^9`. Cần quy về `r = nums[i] % k` trước khi tính khoảng cách vòng tròn.

## 14. Mẹo & nhận dạng pattern

- **"Biến mọi phần tử về cùng số dư mod k bằng ±1"** → chi phí là **khoảng cách vòng tròn** giữa `v mod k` và đích.
- **"Chọn hai giá trị khác nhau"** → giữ **hai giá trị tốt nhất** (kèm chỉ số) để loại trừ trường hợp trùng, đừng thử `k²` cặp.
- **Tổng nhiều hàm khoảng cách trên vòng tròn** → dùng **độ dốc + mảng hiệu**: mỗi điểm đóng góp một đoạn `+1` rồi một đoạn `−1`.
- Chỉ cần `E[0]` trực tiếp rồi cộng độ dốc: mẫu **"giá trị đầu + hiệu"** thay cho tính từng ô.
- Với mọi bài "đưa về số dư" nên viết brute force dựa trên định nghĩa (tìm số `w ≡ t` gần nhất) trước để kiểm chứng công thức.

## 15. Biến thể

1. **Phiên bản I (ràng buộc nhỏ):** thử mọi cặp `(x, y)` với `k` nhỏ — chính là Cách 2/3 chưa cần tối ưu.
2. **Cho phép `x = y`:** bỏ điều kiện khác nhau → đáp án `min(E) + min(O)`.
3. **Chu kỳ `m` (không chỉ 2):** nhóm theo `i mod m` và mỗi nhóm có số dư đích riêng, đôi một khác nhau → bài phân công (cần Hungarian / DP bitmask khi `m` nhỏ).
4. **Chi phí tăng khác chi phí giảm:** độ dốc thay đổi thành `+a` / `−b`, mảng hiệu vẫn dùng được.

In [20]:
# Biến thể 2: cho phép x == y -> min(E) + min(O)
def solve_allow_equal(nums, k):
    E, O = tables_by_residue(nums, k)
    return min(E) + min(O)

print("cho phép x = y:", solve_allow_equal([1, 1, 1], 3), " (đề gốc, x != y:", Solution().minOperations([1, 1, 1], 3), ")")

cho phép x = y: 0  (đề gốc, x != y: 1 )


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn theo định nghĩa | thử mọi cặp `(x, y)`, mỗi phần tử tìm `w` gần nhất | `O(k² · n · k)` | `O(1)` | kiểm chứng, input rất nhỏ |
| 2. Tổng hợp theo số dư | `E[t] = Σ cnt[r]·circ(r,t)`; chọn cặp bằng hai nhỏ nhất | `O(n + D·k)` | `O(k)` | `n, k ≲ 3000` |
| 3. Mảng hiệu vòng tròn | độ dốc `+1/−1/0`, mỗi số dư `O(1)` | `O(n + k)` | `O(k)` | **lời giải nộp** |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `circ(0, 3, 5)` bằng bao nhiêu? (`k = 5`.)

**Bài 2 (tính tay).** `nums = [3, 3, 3, 3]`, `k = 4`. Đáp án là bao nhiêu? (Gợi ý: cả hai nhóm có số dư `3`; hai nhóm phải khác số dư.)

**Bài 3 (điền TODO).** Hoàn thiện `my_second_min_pick(E, O)` trả về `min_{x ≠ y} E[x] + O[y]` bằng hai giá trị nhỏ nhất của `O`.

**Bài 4 (tự viết lại).** Viết `my_min_operations(nums, k)` theo Cách 3, không nhìn code nộp.

**Bài 5 (biến thể).** Viết `min_ops_triple(nums, k)`: chu kỳ **3** (`i mod 3`), ba số dư đích đôi một khác nhau; vét cạn `k ≤ 5` là đủ.

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_second_min_pick(E, O):
    # TODO 1: a = chỉ số của O nhỏ nhất ; second = O nhỏ nhất trong các chỉ số khác a
    # TODO 2: với mỗi x: nếu x == a dùng second, ngược lại dùng O[a]
    # TODO 3: trả về min của E[x] + (giá trị đã chọn)
    pass

# Bài 4 — TODO
def my_min_operations(nums, k):
    # TODO: đếm số dư theo nhóm, mảng hiệu vòng tròn dựng E và O, rồi gọi my_second_min_pick
    pass

# Bài 5 — TODO
def min_ops_triple(nums, k):
    # TODO: vét cạn mọi bộ ba (x, y, z) đôi một khác nhau, chi phí = tổng circ theo i % 3
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `d = (0 − 3) % 5`.
- Mức 2: `d = 2`, `k − d = 3`.
- Mức 3: `min(2, 3) = 2`.

**Bài 2.**
- Mức 1: số dư mọi phần tử là `3`.
- Mức 2: `E[3] = 0`, `O[3] = 0` nhưng cặp `(3, 3)` bị cấm. Phương án rẻ nhất: đổi nhóm lẻ (hai phần tử) lên/xuống 1 bước, hoặc đổi nhóm chẵn.
- Mức 3: mỗi phần tử đổi `1` bước, hai phần tử → `2`.

**Bài 3.**
- Mức 1: tìm `a = argmin O`.
- Mức 2: second = `min(O[y] for y != a)`.
- Mức 3: `min(E[x] + (second if x == a else O[a]) for x in range(k))`.

**Bài 4.**
- Mức 1: xem lại `table_from_counts`.
- Mức 2: `P = k // 2`, hai lần `circular_range_add`.
- Mức 3: `E[t+1] = E[t] + slope` sau khi `slope += D[t]`.

**Bài 5.**
- Mức 1: ba nhóm `i % 3 == 0, 1, 2`.
- Mức 2: `itertools.permutations(range(k), 3)`.
- Mức 3: cộng `circ(v % k, đích_của_nhóm, k)`.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert circ(0, 3, 5) == 2
# Bài 2
assert Solution().minOperations([3, 3, 3, 3], 4) == 2 == brute_definition([3, 3, 3, 3], 4)

# Bài 3
def my_second_min_pick_ans(E, O):
    k = len(E)
    a = min(range(k), key=O.__getitem__)
    second = min(O[y] for y in range(k) if y != a)
    return min(E[x] + (second if x == a else O[a]) for x in range(k))

rng = random.Random(13)
for _ in range(100):
    k = rng.randint(2, 8)
    E = [rng.randint(0, 9) for _ in range(k)]
    O = [rng.randint(0, 9) for _ in range(k)]
    assert my_second_min_pick_ans(E, O) == min(E[x] + O[y] for x in range(k) for y in range(k) if x != y)
print("Bài 1–3 OK")

Bài 1–3 OK


In [23]:
# Bài 4
def my_min_operations_ans(nums, k):
    cnt = [[0] * k, [0] * k]
    for i, v in enumerate(nums):
        cnt[i & 1][v % k] += 1
    E = table_from_counts(cnt[0], k)
    O = table_from_counts(cnt[1], k)
    return my_second_min_pick_ans(E, O)

rng = random.Random(14)
for _ in range(100):
    k = rng.randint(2, 15); n = rng.randint(1, 20)
    nums = [rng.randint(1, 10**6) for _ in range(n)]
    assert my_min_operations_ans(nums, k) == Solution().minOperations(nums, k)

# Bài 5
from itertools import permutations
def min_ops_triple_ans(nums, k):
    best = None
    for xs in permutations(range(k), 3):
        tot = sum(circ(v % k, xs[i % 3], k) for i, v in enumerate(nums))
        best = tot if best is None else min(best, tot)
    return best

assert min_ops_triple_ans([0, 1, 2], 3) == 0
assert min_ops_triple_ans([1, 1, 1], 3) == 2          # ba nhóm cùng số dư 1, chỉ một nhóm giữ được
assert min_ops_triple_ans([1, 1, 1, 1, 1, 1], 4) == 4
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
P = k // 2
cnt[nhóm][v % k] += 1
def build(c):                       # E[t] cho mọi t
    D = [0]*(k+1); e0 = 0
    for r, m in enumerate(c):
        if m:
            e0 += m * min(r, k - r)
            range_add(D, r, P, +m)                    # +m trên P vị trí từ r
            range_add(D, (r + k - P) % k, P, -m)      # -m trên P vị trí từ r+k-P
    E[0] = e0; E[t+1] = E[t] + prefix(D)[t]
a = argmin(O); second = min(O[y] for y != a)
answer = min(E[x] + (second if x == a else O[a]) for x in range(k))
```

**Checklist trước khi submit:**

- [ ] Chi phí là khoảng cách **vòng tròn** `min(d, k − d)` (không tuyến tính, không chỉ một chiều).
- [ ] Quy về `v % k` trước khi tính.
- [ ] `x ≠ y`: dùng hai giá trị nhỏ nhất, không `min(E) + min(O)`.
- [ ] Mảng hiệu cho đoạn quấn vòng tách thành hai đoạn; `D` dài `k + 1`.
- [ ] `P = k // 2`; vị trí giữa độ dốc `0` khi `k` lẻ.
- [ ] `n = 1`: nhóm lẻ rỗng (`O ≡ 0`), vẫn cần `y ≠ x` (luôn có vì `k ≥ 2`).
- [ ] Đã đối chiếu vét cạn theo định nghĩa với `k ≤ 7`, `n ≤ 7`.
- [ ] Đã đối chiếu chữ ký phương thức Premium trên LeetCode.